# Module 8: Tune and Evaluate

In Module 7 you found the saturation knee on the FP8 deployment and its current speculative setting. This module turns that observation into an operating point. You change one vLLM flag in the shared manifest, redeploy, run the same sweep, and decide whether to keep or reject the change. The goal is not the biggest number. The goal is a defensible configuration for your workload.


## Learning objectives
- Name the vLLM flags that move cache headroom and scheduler behavior
- Start from the FP8 deployment created in Module 5 and the speculative-decode decision from Module 6
- Change one setting in `manifests/vllm.yaml`, redeploy, and wait for readiness
- Compare throughput and latency before and after the change, then keep or reject it based on data
- Account for quality checks before calling the tuning done


## Prerequisites
- Finished Module 7 and recorded the saturation baseline
- Namespace permission to apply your own `deployment/vllm`
- About 30 minutes


References: [vLLM optimization and tuning](https://docs.vllm.ai/en/stable/configuration/optimization/) &middot; [vLLM engine arguments](https://docs.vllm.ai/en/stable/serving/engine_args.html) &middot; [kubectl rollout](https://kubernetes.io/docs/reference/generated/kubectl/kubectl-commands#rollout)


## Tuning design basics

Tuning is a loop, not a paste-in final manifest:

1. Measure the current FP8 deployment and its current speculative-decode setting.
2. Pick one bottleneck from the metrics.
3. Change one flag in `manifests/vllm.yaml`.
4. Apply the manifest and wait for the pod.
5. Run the same measurement again.
6. Keep the change only if the data supports it; otherwise revert and document why.

Changing many flags at once hides the cause.

![A tuning loop that measures the current deployment, changes one manifest flag, applies the rollout, remeasures, and records the operating point](images/08_tune_and_evaluate_architecture.png)


## 1. Setup

Resolve your settings and the shared manifest path. The path logic matches Module 5, so the notebook works from the module folder or the repo root.


In [ ]:

%pip install -q "openai>=1.40" "requests>=2.31"


In [ ]:

# Imports, settings, and the shared manifest path.
import os, sys
from pathlib import Path

if Path("../manifests/vllm.yaml").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")

sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import print_settings
from common import loadtest

settings = print_settings()
MANIFEST = REPO_ROOT / "manifests" / "vllm.yaml"
TARGET_MODEL = "RedHatAI/Qwen3-4B-FP8-dynamic"
print("manifest:", MANIFEST)
print("target  :", TARGET_MODEL)

import requests
root = settings.vllm_host.rstrip("/").removesuffix("/v1")

def served_models():
    data = requests.get(
        f"{root}/v1/models",
        headers={"Authorization": f"Bearer {settings.api_key}"},
        timeout=20,
    ).json()
    return [item["id"] for item in data.get("data", [])]

models = served_models()
print("served models:", models)
assert TARGET_MODEL in models, f"Expected {TARGET_MODEL}; got {models}"


**What you should see:** your endpoint settings, the shared manifest path, the FP8 target model id, and that model in the served-model list. If the manifest is missing, finish Module 5's shared-manifest setup first.


## 2. The knobs

Start with the flags already in `manifests/vllm.yaml`:

- `--gpu-memory-utilization`: how much GPU memory vLLM may reserve. More can mean more KV cache, but too high can fail startup.
- `--max-model-len`: the context length the engine plans around. Larger context costs cache capacity.
- `--max-num-seqs`: the maximum number of requests in the running batch.
- `--max-num-batched-tokens`: the token budget per scheduler step across prefill and decode.
- KV-cache dtype, when available: can trade cache precision for more cache headroom.

Pick the next knob from the bottleneck you saw in Module 7.


## 3. Measure the current baseline

Run the same sweep shape you used in Module 7. This is the before row for your tuning table and the baseline you may choose to keep. Use the target model id explicitly; the `MODEL_NAME` environment variable may not reflect the manual manifest edits.


In [ ]:

# Requires a live vLLM endpoint.
levels = [1, 8, 32, 64, 128]
baseline = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)
baseline


**What you should see:** the same general knee as Module 7. If the numbers changed a lot, check whether the manifest, model, or workload shape changed.


## 4. Change one thing

Open `manifests/vllm.yaml` and change one value. A conservative first path is:

- If KV cache was tight: raise `--gpu-memory-utilization` from the manifest's current value, such as `0.7`, toward `0.8` or `0.9`.
- If throughput flattened and latency was too high: cap the hot operating point with `--max-num-seqs` so the server protects p95 instead of accepting unlimited concurrency.
- If long prompts waited too long: adjust `--max-num-batched-tokens` and watch whether TTFT improves without hurting throughput.

Do not change the model here. If speculative decoding hurt in Module 6, record that decision; Module 8 is about serving-policy knobs, not changing the model family.


In [ ]:

# Requires a live cluster. Preview and apply the one manifest edit.
ns = settings.namespace
!kubectl diff -n {ns} -f {MANIFEST}
!kubectl apply -n {ns} -f {MANIFEST}
!kubectl rollout status -n {ns} deploy/vllm --timeout=10m


**What you should see:** `kubectl diff` should show the single flag value you changed, not an accidental model or image change. `rollout status` should finish successfully. If the pod never becomes Ready, the new memory setting may be too aggressive; inspect the pod logs and back it off.


## 5. Re-measure and compare

Run the exact same sweep. A faster run with different prompt length, output length, or concurrency levels does not prove the tuning helped.


In [ ]:

# Requires the redeployed endpoint to be Ready.
after = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)
after


In [ ]:

# Print a compact before/after table.
print("concurrency | before tok/s | after tok/s | before p95 TTFT | after p95 TTFT")
for b, a in zip(baseline, after):
    print(f"{b['concurrency']:>11} | {b['throughput_tok_s']:>12} | {a['throughput_tok_s']:>10} | "
          f"{b['ttft_p95_ms']:>15} | {a['ttft_p95_ms']:>13}")

best_before = max(row["throughput_tok_s"] for row in baseline)
best_after = max(row["throughput_tok_s"] for row in after)
print(f"peak throughput: {best_before:.1f} -> {best_after:.1f} tok/s ({best_after / best_before:.2f}x)")


**What you should see:** a table that makes the tradeoff visible. A higher peak is useful only if the latency at your operating point still meets the bar. If the numbers do not improve, the correct outcome is to reject the change and keep the baseline.


## 6. Bring quality back

Speed is not the whole operating point. Scheduler and cache flags should not usually change answers, but production changes are not complete until quality is checked. For the workshop, record the performance operating point. For production, pair that record with workload evals before you keep the final settings.


## Things to know

- **One change, one measurement.** Otherwise you cannot explain the result.
- **The model stays fixed.** Module 8 tunes scheduler and cache flags on top of the deployment state you chose after Modules 5 and 6.
- **A higher peak can be worse.** If p95 latency is unacceptable, the operating point is too hot.
- **Record the values.** Your final artifact is model, precision, flag values, workload shape, throughput, latency, and quality result.


## Try it yourself

**Tune the batch cap.** Raise `--max-num-seqs`, redeploy, and run the same sweep. Stop when throughput stops moving or latency rises too much.

**Trade context for concurrency.** Lower `--max-model-len` and watch whether more concurrent requests fit. Only keep that change if your application can live with the shorter context.


## Summary

- Module 8 is the measure, change, redeploy, re-measure, keep-or-reject loop.
- The main flags move cache headroom, context planning, and scheduler batch size.
- The best configuration is an operating point that balances throughput, latency, cost, and quality.
- You now have a measured FP8 vLLM operating point that later applications can sit on top of.


## Next

**Hand back to Du'An, Module 9: Agents on Kubernetes.** The optional capstone puts a small agent service on top of the inference stack whose operating point you just measured and keeps the same operational discipline: deploy, call it, and watch the inference metrics move.
